In [1]:

from archetypes_re2020.utils.utils import *
import archetypes_re2020.utils.bw25_utils as bw25u
from archetypes_re2020.data.source import load_parquet_data
from archetypes_re2020.config import INTERIM_DATA_DIR
from archetypes_re2020.buildings.buildings_db import *
from archetypes_re2020.config import RAW_DATA_DIR, INTERIM_DATA_DIR

import numpy as np
import pandas as pd
import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import warnings

import bw2data as bd
import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa

from datetime import datetime
import os
import re
import openpyxl
import math
import questionary
from tqdm import tqdm

10:54:27+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [3]:
list(bd.databases)

['ciogen_custom',
 'ciogen_fin_de_vie',
 'pr_based_arch_alamain',
 'ciogen_production',
 '2026-3-23-15h9_archetypes_RE2020_BB',
 '2026-3-23-14h21_buildings_RE2020',
 'ecoinvent-3.11-cutoff',
 '2026-3-19-11h18_processes_RE2020',
 'ciogen_transport',
 'ecoinvent-3.11-biosphere',
 'ciogen_ponts_production',
 '2026-3-27_energy_processes']

In [10]:
del bd.databases['2026-2-12-10h51_bernoulli_RE2020']

12:09:40+0100 [info     ] Vacuuming database            


In [4]:

# Constantes
INPUT_PATH = INTERIM_DATA_DIR / "quantites_augmentees.parquet"

DEFAULT_BUILDING_UNIT = "m2"
DEFAULT_LOCATION = "FR"
DATABASE_NAME_PATTERN = "{year}-{month}-{day}-{hour}h{minute}_batiments_RE2020"


In [4]:
df_buildings = pl.read_csv(
    source=RAW_DATA_DIR / "202505_export_batiments_daact.csv", ignore_errors=True
    )

In [ ]:
df_buildings.filter(pl.col("batiment_index")==1,pl.col("projet_id")== 354459)

113.09

In [5]:
df_quantities = load_parquet_data(INPUT_PATH)

In [ ]:
d = {a:1}

In [8]:
df_buildings.columns

['altitude',
 'art_45',
 'batiment_conforme_re2020',
 'batiment_dc_id',
 'batiment_index',
 'batiment_rsenv_entree_projet_id',
 'batiment_rsenv_sortie_projet_id',
 'batiment_rset_entree_projet_id',
 'batiment_rset_sortie_projet_id',
 'q4pa_sparoi',
 'q4pa_sref',
 'brasseur_air_nb',
 'brasseur_air_rat_surf',
 'brasseur_air_qv_air_max',
 'bbio_ch_pts',
 'bbio_conforme',
 'bbio_ecl_pts',
 'bbio_fr_pts',
 'bbio_gain',
 'bbio_max',
 'bbio_projet',
 'capteur_pv_pc',
 'capteur_pv_surface',
 'categorie_ce',
 'cef_bois',
 'cef_bois_ch',
 'cef_bois_ecs',
 'cef_ch',
 'cef_depl',
 'cef_dist',
 'cef_ecl',
 'cef_ecs',
 'cef_elec',
 'cef_elec_ch',
 'cef_elec_depl',
 'cef_elec_dist',
 'cef_elec_ecl',
 'cef_elec_ecs',
 'cef_elec_fr',
 'cef_elec_mobi',
 'cef_elec_vent',
 'cef_fr',
 'cef_gaz',
 'cef_gaz_ch',
 'cef_gaz_ecs',
 'cef_gaz_fr',
 'cef_mobi',
 'cef_projet',
 'cef_reseau',
 'cef_reseau_ch',
 'cef_reseau_ecs',
 'cef_reseau_fr',
 'cef_vent',
 'cep_conforme',
 'cep_gain',
 'cep_max',
 'cep_nr_confor

In [6]:
df_quantities.columns

['projet_id',
 'batiment_index',
 'zone_index',
 'lot_ref',
 'sous_lot_ref',
 'type_donnees',
 'type_declaration',
 'unite_uf_txt',
 'nomenclature_inies_1',
 'nomenclature_inies_2',
 'nomenclature_inies_3',
 'nomenclature_inies_4',
 'nomenclature_inies_5',
 'dve',
 'quantite',
 'nomenclatures',
 'usage_principal_1_txt',
 'sref',
 'puissance_thermique_max_restituable_ch_1',
 'puissance_thermique_max_restituable_ch_2',
 'quant_surf',
 'nomenclatures_UF',
 'nomenclatures_UF_dve']

In [4]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [5]:
bd.databases

Databases dictionary with 13 object(s):
	2026-2-5-19h15_procedes_RE2020
	2026-2-6-10h22_archetypes_RE2020
	2026-2-6-10h22_bernoulli_RE2020
	2026-2-9-11h40_batiments_RE2020
	Condorcet_db
	bernou_test
	ciogen_custom
	ciogen_fin_de_vie
	ciogen_ponts_production
	ciogen_production
	ciogen_transport
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff

In [6]:
eidb = bd.Database("ecoinvent-3.11-cutoff")
biodb = bd.Database("ecoinvent-3.11-biosphere")
prodb = bd.Database("2026-2-5-19h15_procedes_RE2020")
name_db = "bat_test_db"

In [ ]:
#batdb = bd.Database(name_db)
    
    # Ajout des codes produits
df_quantities = add_product_codes(df_quantities, prodb)

In [ ]:
df_quantities.columns

In [ ]:
df_quantities.filter(pl.col("code_prodb") != "False")

In [ ]:
a = prodb.random()

In [ ]:
prod_set = {act['code'] for act in prodb}

In [ ]:
prod_set